In [ ]:
# ================================================================
# SINGLE-CELL IMPLEMENTATION
# Evidence-Grounded Multimodal Retrieval-Augmented Foundation Models
# for Scientific Knowledge Discovery and Uncertainty-Aware Reasoning
#
# Colab usage:
# 1) Set DATA_DIR to a folder containing scientific PDFs (or upload PDFs).
# 2) Optionally set SPIQA_JSONL to a SPIQA question/answer JSONL file.
# 3) Run this cell. First run downloads model weights and may take time.
# 4) Ask questions with ask_scientific_question("your question").
#
# The pipeline implements:
# PyMuPDF parsing + layout segmentation + text/figure/table extraction;
# BGE-M3 + CLIP embeddings + FAISS;
# BM25 + dense/visual retrieval + RRF + cross-encoder reranking (CMSR);
# SciBERT-based entity representation + relation extraction + graph/GAT (ESKG);
# Qwen2.5-VL evidence-conditioned generation + NLI + temperature scaling (ECMG);
# optional SPIQA benchmark evaluation and ablation.
#
# Important: this is a runnable research implementation. Model outputs and
# metrics must be evaluated on your real corpus; no target scores are fabricated.
# ================================================================

import os, sys, re, json, math, glob, gc, warnings, subprocess, importlib.util, hashlib, random
from pathlib import Path
from collections import defaultdict, Counter
warnings.filterwarnings("ignore")

# ------------------------- USER CONFIG ----------------------------
DATA_DIR = "/content/scientific_papers"       # folder containing PDFs
OUTPUT_DIR = "/content/EG_Multimodal_RAG_Outputs"
SPIQA_JSONL = ""                              # optional path to SPIQA QA JSONL
MAX_PDFS = None                               # None = all PDFs
MAX_PAGES_PER_PDF = None                      # None = all pages
TOP_K_SPARSE = 15
TOP_K_DENSE = 15
TOP_K_VISUAL = 8
TOP_K_RERANK = 10
MAX_CONTEXT_CHARS = 14000
USE_QWEN_VL = True
USE_GPU = True
RANDOM_SEED = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)

# ---------------------- INSTALL DEPENDENCIES ----------------------
def _pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

required = {
    "fitz": "pymupdf",
    "sentence_transformers": "sentence-transformers",
    "transformers": "transformers",
    "faiss": "faiss-cpu",
    "rank_bm25": "rank-bm25",
    "PIL": "pillow",
    "sklearn": "scikit-learn",
    "networkx": "networkx",
    "pandas": "pandas",
    "matplotlib": "matplotlib"
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", ", ".join(missing))
    _pip_install(missing)
# PyTorch is preinstalled in Colab in most runtimes. Install only if absent.
if importlib.util.find_spec("torch") is None:
    _pip_install(["torch", "torchvision"])

import numpy as np
import pandas as pd
import fitz
import torch
import torch.nn as nn
import torch.nn.functional as F
import faiss
import networkx as nx
from PIL import Image
from rank_bm25 import BM25Okapi
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix,
                             roc_auc_score, log_loss)
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          pipeline)
import matplotlib.pyplot as plt

DEVICE = "cuda" if (USE_GPU and torch.cuda.is_available()) else "cpu"
print("Device:", DEVICE)

# ------------------- STAGE 1: DOCUMENT PREPARATION ----------------
def clean_text(text):
    text = (text or "").replace("\x00", " ")
    text = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

SECTION_PATTERNS = [
    ("abstract", r"^\s*(abstract|summary)\s*$"),
    ("introduction", r"^\s*(1\.?\s*)?introduction\s*$"),
    ("methodology", r"^\s*(materials and methods|methods|methodology|experimental setup)\s*$"),
    ("results", r"^\s*(results|experimental results)\s*$"),
    ("discussion", r"^\s*discussion\s*$"),
    ("conclusion", r"^\s*(conclusions?|future work)\s*$"),
    ("references", r"^\s*references\s*$")
]

def detect_section(text, current="body"):
    for name, pat in SECTION_PATTERNS:
        if re.match(pat, (text or "").strip(), flags=re.I):
            return name
    return current

def extract_pdf(pdf_path):
    doc_id = hashlib.md5(str(pdf_path).encode()).hexdigest()[:12]
    records, figures = [], []
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print("PDF open failed:", pdf_path, e)
        return records, figures
    metadata = doc.metadata or {}
    section = "body"
    pages = range(len(doc))
    if MAX_PAGES_PER_PDF:
        pages = range(min(len(doc), MAX_PAGES_PER_PDF))
    for pno in pages:
        page = doc[pno]
        raw = page.get_text("dict")
        blocks = []
        for block in raw.get("blocks", []):
            if block.get("type") != 0:
                continue
            lines = []
            for line in block.get("lines", []):
                spans = line.get("spans", [])
                line_text = "".join(s.get("text", "") for s in spans).strip()
                if line_text:
                    lines.append(line_text)
            txt = clean_text("\n".join(lines))
            if not txt:
                continue
            # Heuristic layout-aware segmentation: font size/boldness + heading patterns.
            spans = [s for b in block.get("lines", []) for s in b.get("spans", [])]
            max_size = max([s.get("size", 0) for s in spans] or [0])
            is_bold = any("bold" in s.get("font", "").lower() for s in spans)
            is_heading = (len(txt) < 120 and (max_size >= 12 or is_bold or
                           re.match(r"^(\d+(\.\d+)*\.?\s+)?[A-Z][A-Za-z \-]{2,60}$", txt)))
            if is_heading:
                section = detect_section(txt, section)
                if section == "body":
                    section = "heading:" + txt[:70]
            else:
                section = detect_section(txt, section)
            blocks.append((txt, section))
        page_text = "\n".join(x[0] for x in blocks)
        for bi, (txt, sec) in enumerate(blocks):
            if len(txt) < 25:
                continue
            records.append({
                "doc_id": doc_id, "document": Path(pdf_path).name,
                "page": pno + 1, "block_id": bi, "section": sec,
                "text": txt, "metadata_title": metadata.get("title", "") or Path(pdf_path).stem,
                "source_type": "text"
            })
        # Extract embedded images; retain source page and image index.
        image_list = page.get_images(full=True)
        for img_i, img_info in enumerate(image_list):
            xref = img_info[0]
            try:
                pix = fitz.Pixmap(doc, xref)
                if pix.n - pix.alpha > 3:
                    pix = fitz.Pixmap(fitz.csRGB, pix)
                image_path = os.path.join(OUTPUT_DIR, f"{doc_id}_p{pno+1}_img{img_i+1}.png")
                pix.save(image_path)
                if os.path.getsize(image_path) > 1500:
                    figures.append({
                        "doc_id": doc_id, "document": Path(pdf_path).name,
                        "page": pno + 1, "figure_id": f"{doc_id}_p{pno+1}_fig{img_i+1}",
                        "image_path": image_path, "caption": "",
                        "source_type": "figure"
                    })
            except Exception:
                pass
        # PyMuPDF table extraction when available; table rows stored as textual evidence.
        try:
            tabs = page.find_tables()
            for ti, tab in enumerate(tabs.tables):
                data = tab.extract()
                if data:
                    table_text = "\n".join(" | ".join("" if c is None else str(c) for c in row)
                                           for row in data)
                    if table_text.strip():
                        records.append({
                            "doc_id": doc_id, "document": Path(pdf_path).name,
                            "page": pno + 1, "block_id": 10000 + ti,
                            "section": "table", "text": "TABLE:\n" + table_text,
                            "metadata_title": metadata.get("title", "") or Path(pdf_path).stem,
                            "source_type": "table"
                        })
        except Exception:
            pass
    doc.close()
    # Associate likely captions using nearby text blocks on the same page.
    for fig in figures:
        nearby = [r["text"] for r in records if r["doc_id"] == fig["doc_id"]
                  and r["page"] == fig["page"]
                  and re.search(r"\b(fig(ure)?|chart|scheme)\s*\d+", r["text"], re.I)]
        fig["caption"] = " ".join(nearby[:2])
    return records, figures

pdf_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.pdf"), recursive=True))
if MAX_PDFS:
    pdf_files = pdf_files[:MAX_PDFS]
if not pdf_files:
    print(f"WARNING: No PDFs found in {DATA_DIR}. Create the folder and add scientific PDFs.")
    print("You can still run the remaining code after adding PDFs and rerunning this cell.")
all_records, all_figures = [], []
for i, pdf in enumerate(pdf_files, 1):
    print(f"[1/6] Parsing {i}/{len(pdf_files)}: {Path(pdf).name}")
    rec, figs = extract_pdf(pdf)
    all_records.extend(rec); all_figures.extend(figs)

# Split long text into overlapping chunks while preserving provenance.
def make_chunks(records, chunk_words=220, overlap=40):
    chunks = []
    for r in records:
        words = r["text"].split()
        if len(words) <= chunk_words:
            pieces = [r["text"]]
        else:
            pieces = []
            step = max(1, chunk_words - overlap)
            for st in range(0, len(words), step):
                part = " ".join(words[st:st+chunk_words])
                if len(part.split()) >= 25:
                    pieces.append(part)
        for ci, part in enumerate(pieces):
            item = dict(r)
            item["chunk_id"] = f'{r["doc_id"]}_p{r["page"]}_b{r["block_id"]}_c{ci}'
            item["text"] = part
            chunks.append(item)
    return chunks

chunks = make_chunks(all_records)
pd.DataFrame(all_records).to_json(os.path.join(OUTPUT_DIR, "structured_document_repository.jsonl"),
                                  orient="records", lines=True, force_ascii=False)
pd.DataFrame(all_figures).to_csv(os.path.join(OUTPUT_DIR, "figure_repository.csv"), index=False)
print("Structured passages:", len(chunks), "| Figures:", len(all_figures))

# ---------------- STAGE 2: MULTIMODAL EMBEDDINGS + FAISS -----------
print("[2/6] Loading BGE-M3 and CLIP encoders...")
TEXT_MODEL_NAME = "BAAI/bge-m3"
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
text_encoder = SentenceTransformer(TEXT_MODEL_NAME, device=DEVICE)
from transformers import CLIPModel, CLIPProcessor
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

passages = [c["text"] for c in chunks]
if passages:
    text_embeddings = text_encoder.encode(passages, batch_size=16, show_progress_bar=True,
                                          normalize_embeddings=True).astype("float32")
    text_index = faiss.IndexFlatIP(text_embeddings.shape[1])
    text_index.add(text_embeddings)
else:
    text_embeddings = np.zeros((0, 1024), dtype="float32")
    text_index = None

def clip_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
        emb = F.normalize(emb, p=2, dim=-1)
    return emb.cpu().numpy()[0].astype("float32")

def clip_text_embedding(text):
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True,
                            truncation=True).to(DEVICE)
    with torch.no_grad():
        emb = clip_model.get_text_features(**inputs)
        emb = F.normalize(emb, p=2, dim=-1)
    return emb.cpu().numpy()[0].astype("float32")

figure_embeddings = []
valid_figures = []
for f in all_figures:
    try:
        figure_embeddings.append(clip_image_embedding(f["image_path"]))
        valid_figures.append(f)
    except Exception as e:
        print("Skipping image embedding:", f["image_path"], str(e)[:100])
if figure_embeddings:
    figure_embeddings = np.vstack(figure_embeddings).astype("float32")
    visual_index = faiss.IndexFlatIP(figure_embeddings.shape[1])
    visual_index.add(figure_embeddings)
else:
    figure_embeddings = np.zeros((0, 512), dtype="float32")
    visual_index = None

# Persist vectors and metadata for reuse.
if len(text_embeddings):
    np.save(os.path.join(OUTPUT_DIR, "bge_m3_embeddings.npy"), text_embeddings)
if len(figure_embeddings):
    np.save(os.path.join(OUTPUT_DIR, "clip_figure_embeddings.npy"), figure_embeddings)
print("Text vectors:", text_embeddings.shape, "| Visual vectors:", figure_embeddings.shape)

# ---------------- STAGE 3: CMSR RETRIEVAL -------------------------
print("[3/6] Preparing sparse and reranking models...")
tokenized_corpus = [re.findall(r"\w+", p.lower()) for p in passages]
bm25 = BM25Okapi(tokenized_corpus) if tokenized_corpus else None
try:
    reranker = CrossEncoder("BAAI/bge-reranker-base", device=DEVICE)
except Exception as e:
    print("Cross-encoder unavailable; using dense scores for reranking:", str(e)[:150])
    reranker = None

def reciprocal_rank_fusion(rank_lists, k=60):
    scores = defaultdict(float)
    for ranking in rank_lists:
        for rank, idx in enumerate(ranking, 1):
            scores[idx] += 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

def retrieve_evidence(query, top_sparse=TOP_K_SPARSE, top_dense=TOP_K_DENSE,
                      top_visual=TOP_K_VISUAL, top_final=TOP_K_RERANK):
    if not chunks:
        return [], []
    sparse_ids, dense_ids, visual_ids = [], [], []
    if bm25 is not None:
        q_tokens = re.findall(r"\w+", query.lower())
        sparse_scores = bm25.get_scores(q_tokens)
        sparse_ids = np.argsort(sparse_scores)[::-1][:top_sparse].tolist()
    qemb = text_encoder.encode([query], normalize_embeddings=True).astype("float32")
    if text_index is not None:
        _, ids = text_index.search(qemb, min(top_dense, len(chunks)))
        dense_ids = [int(i) for i in ids[0] if i >= 0]
    if visual_index is not None:
        try:
            qv = clip_text_embedding(query).reshape(1, -1)
            _, vids = visual_index.search(qv, min(top_visual, len(valid_figures)))
            visual_ids = [int(i) for i in vids[0] if i >= 0]
        except Exception as e:
            print("Visual retrieval skipped:", str(e)[:100])
    fused = reciprocal_rank_fusion([sparse_ids, dense_ids])
    candidates = fused[:max(top_final * 3, top_final)]
    if reranker is not None and candidates:
        pairs = [(query, chunks[i]["text"]) for i in candidates]
        try:
            scores = reranker.predict(pairs, batch_size=16)
            candidates = [x for _, x in sorted(zip(scores, candidates), reverse=True)]
        except Exception:
            pass
    selected = candidates[:top_final]
    text_evidence = []
    for idx in selected:
        c = chunks[idx]
        text_evidence.append({
            "text": c["text"], "document": c["document"], "page": c["page"],
            "section": c["section"], "chunk_id": c["chunk_id"],
            "source_type": c["source_type"], "score_rank": len(text_evidence)+1
        })
    visual_evidence = []
    for vi in visual_ids:
        if vi < len(valid_figures):
            f = valid_figures[vi]
            visual_evidence.append({**f, "similarity_rank": len(visual_evidence)+1})
    return text_evidence, visual_evidence

# ---------------- STAGE 4: ESKG + SCIPERT + GAT --------------------
print("[4/6] Loading SciBERT and constructing evidence graph...")
SCIBERT_NAME = "allenai/scibert_scivocab_uncased"
sci_tokenizer = AutoTokenizer.from_pretrained(SCIBERT_NAME)
sci_model = AutoModel.from_pretrained(SCIBERT_NAME).to(DEVICE).eval()

# SciBERT contextual token representations + noun/technical phrase candidates.
# This is a transparent unsupervised entity candidate extractor; replace with
# a SciBERT fine-tuned scientific NER checkpoint for domain-specific NER labels.
STOP = set("""the a an and or of to in for on with by from is are was were be as that this
these those using use used based proposed method methods model models result results
we our their it its can may""".split())
def extract_scientific_entities(text, max_entities=18):
    tokens = re.findall(r"\b[A-Za-z][A-Za-z0-9\-]{2,}(?:\s+[A-Za-z][A-Za-z0-9\-]{2,}){0,2}\b", text)
    candidates = []
    for t in tokens:
        norm = re.sub(r"\s+", " ", t).strip()
        if norm.lower() not in STOP and len(norm) > 3:
            candidates.append(norm)
    counts = Counter(candidates)
    return [x for x, _ in counts.most_common(max_entities)]

RELATION_PATTERNS = {
    "uses": r"\b(uses?|using|utilizes?|employs?)\b",
    "evaluates": r"\b(evaluates?|assesses?|measures?)\b",
    "improves": r"\b(improves?|enhances?|increases?|outperforms?)\b",
    "compares": r"\b(compares?|compared with|versus|vs\.?)\b",
    "contradicts": r"\b(contradicts?|conflicts with|inconsistent with)\b",
    "supports": r"\b(supports?|demonstrates?|shows?|indicates?)\b"
}

def build_evidence_graph(evidence):
    G = nx.MultiDiGraph()
    for ei, ev in enumerate(evidence):
        source = f'{ev["document"]}:p{ev["page"]}'
        G.add_node(source, node_type="source", provenance=source)
        ents = extract_scientific_entities(ev["text"])
        for ent in ents:
            G.add_node(ent, node_type="scientific_entity")
            G.add_edge(source, ent, relation="mentions", provenance=source)
        # Extract relation candidates from nearby entity pairs and sentence cues.
        sentences = re.split(r"(?<=[.!?])\s+", ev["text"])
        for sent in sentences:
            se = extract_scientific_entities(sent, max_entities=8)
            if len(se) >= 2:
                relation = next((r for r, pat in RELATION_PATTERNS.items()
                                 if re.search(pat, sent, re.I)), "related_to")
                G.add_edge(se[0], se[1], relation=relation, evidence=sent[:500],
                           provenance=source)
    return G

class EvidenceGAT(nn.Module):
    """Small attention-based graph message-passing network (PyTorch implementation)."""
    def __init__(self, in_dim=128, hidden=64, out_dim=32):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden)
        self.attn = nn.Linear(hidden * 2, 1)
        self.out = nn.Linear(hidden, out_dim)
    def forward(self, x, edge_index):
        h = torch.tanh(self.proj(x))
        if edge_index.numel() == 0:
            return self.out(h)
        src, dst = edge_index
        pair = torch.cat([h[src], h[dst]], dim=1)
        alpha = torch.sigmoid(self.attn(pair)).squeeze(-1)
        agg = torch.zeros_like(h)
        deg = torch.zeros((h.size(0), 1), device=h.device)
        agg.index_add_(0, dst, h[src] * alpha.unsqueeze(1))
        deg.index_add_(0, dst, alpha.unsqueeze(1))
        h2 = h + agg / deg.clamp_min(1e-6)
        return self.out(h2)

def graph_attention_summary(G):
    nodes = list(G.nodes())
    if not nodes:
        return {"nodes": [], "edges": [], "important_entities": []}
    node_to_id = {n:i for i,n in enumerate(nodes)}
    # Use BGE embeddings as node features; project to a fixed 128 dimensions.
    names = [str(n) for n in nodes]
    embs = text_encoder.encode(names, normalize_embeddings=True).astype("float32")
    if embs.shape[1] < 128:
        embs = np.pad(embs, ((0,0),(0,128-embs.shape[1])))
    x = torch.tensor(embs[:, :128], dtype=torch.float32, device=DEVICE)
    edges = [(node_to_id[u], node_to_id[v]) for u,v in G.edges()]
    edge_index = torch.tensor(edges, dtype=torch.long, device=DEVICE).T.contiguous() if edges else torch.empty((2,0), dtype=torch.long, device=DEVICE)
    gat = EvidenceGAT().to(DEVICE).eval()
    with torch.no_grad():
        z = gat(x, edge_index).cpu().numpy()
    centrality = nx.degree_centrality(G)
    important = sorted([(n, float(centrality.get(n,0))) for n in nodes
                        if G.nodes[n].get("node_type") == "scientific_entity"],
                       key=lambda x:x[1], reverse=True)[:12]
    return {
        "nodes": [{"id": str(n), **G.nodes[n]} for n in nodes],
        "edges": [{"source": str(u), "target": str(v), **d}
                  for u,v,d in G.edges(data=True)],
        "important_entities": [{"entity": n, "graph_score": s} for n,s in important],
        "gat_embedding_shape": list(z.shape)
    }

# ---------------- STAGE 5: ECMG, NLI, CALIBRATION -----------------
print("[5/6] Loading NLI verifier and optional Qwen2.5-VL generator...")
NLI_NAME = "cross-encoder/nli-deberta-v3-base"
try:
    nli_tokenizer = AutoTokenizer.from_pretrained(NLI_NAME)
    nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_NAME).to(DEVICE).eval()
    nli_labels = {int(v): k.lower() for k,v in nli_model.config.id2label.items()}
except Exception as e:
    print("NLI model unavailable; lexical support check will be used:", str(e)[:180])
    nli_tokenizer = nli_model = None
    nli_labels = {}

qwen_model = qwen_processor = None
if USE_QWEN_VL:
    try:
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
        qwen_name = "Qwen/Qwen2.5-VL-3B-Instruct"
        qwen_processor = AutoProcessor.from_pretrained(qwen_name)
        qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            qwen_name, torch_dtype=(torch.float16 if DEVICE=="cuda" else torch.float32),
            device_map="auto" if DEVICE=="cuda" else None
        )
        if DEVICE != "cuda":
            qwen_model = qwen_model.to(DEVICE)
        qwen_model.eval()
    except Exception as e:
        print("Qwen2.5-VL could not be loaded; generation fallback will be extractive:", str(e)[:250])
        qwen_model = qwen_processor = None

def make_evidence_context(text_evidence, visual_evidence, graph_summary):
    parts = []
    for i, ev in enumerate(text_evidence, 1):
        parts.append(f"[E{i}] {ev['document']} | page {ev['page']} | {ev['section']}\n{ev['text']}")
    if graph_summary.get("important_entities"):
        parts.append("[GRAPH ENTITIES] " + ", ".join(x["entity"] for x in graph_summary["important_entities"]))
    if visual_evidence:
        parts.append("[VISUAL EVIDENCE] " + "; ".join(
            f"{v['document']} page {v['page']} figure {v['figure_id']} caption: {v.get('caption','')}"
            for v in visual_evidence))
    return "\n\n".join(parts)[:MAX_CONTEXT_CHARS]

def qwen_generate(query, context, visual_evidence):
    if qwen_model is None:
        # Faithful fallback: quote evidence snippets rather than inventing a model answer.
        snippets = [x.strip() for x in re.split(r"(?<=[.!?])\s+", context) if len(x.strip()) > 60][:5]
        return ("The available retrieved evidence indicates:\n" +
                "\n".join("- " + s[:500] for s in snippets) if snippets else
                "Insufficient evidence was retrieved to answer this question.")
    try:
        content = [{"type":"text", "text":
            "Answer the scientific question using ONLY the supplied evidence. "
            "Do not invent facts. Distinguish findings from interpretation. "
            "Cite evidence IDs such as [E1]. If evidence is insufficient or conflicting, "
            "state this explicitly.\n\nQUESTION:\n"+query+"\n\nEVIDENCE:\n"+context}]
        # Include top retrieved figures when supported by the installed processor.
        for v in visual_evidence[:2]:
            content.append({"type":"image", "image": v["image_path"]})
        messages = [{"role":"user", "content":content}]
        prompt = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs = None
        try:
            from qwen_vl_utils import process_vision_info
            image_inputs, video_inputs = process_vision_info(messages)
        except Exception:
            pass
        inputs = qwen_processor(text=[prompt], images=image_inputs, padding=True,
                                return_tensors="pt").to(qwen_model.device)
        with torch.no_grad():
            out = qwen_model.generate(**inputs, max_new_tokens=650, do_sample=False)
        generated = out[:, inputs.input_ids.shape[1]:]
        return qwen_processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
    except Exception as e:
        print("Qwen generation fallback:", str(e)[:160])
        return "Generation failed for this run. Retrieved evidence is available in the returned result."

def verify_claims(answer, evidence_texts):
    claims = [s.strip() for s in re.split(r"(?<=[.!?])\s+", answer) if len(s.strip()) > 25]
    evid = " ".join(evidence_texts)
    results = []
    for claim in claims:
        if nli_model is not None:
            try:
                enc = nli_tokenizer(claim, evid[:2500], return_tensors="pt",
                                    truncation=True, max_length=512).to(DEVICE)
                with torch.no_grad():
                    logits = nli_model(**enc).logits[0]
                    probs = torch.softmax(logits, dim=-1).cpu().numpy()
                label_idx = int(np.argmax(probs))
                label = nli_labels.get(label_idx, str(label_idx))
                results.append({"claim": claim, "nli_label": label,
                                "nli_confidence": float(probs[label_idx])})
                continue
            except Exception:
                pass
        # Fallback is explicitly a lexical overlap heuristic, not NLI.
        cw = set(re.findall(r"\w+", claim.lower())) - STOP
        ew = set(re.findall(r"\w+", evid.lower())) - STOP
        overlap = len(cw & ew) / max(1, len(cw))
        results.append({"claim": claim, "nli_label": "lexical_support_heuristic",
                        "nli_confidence": float(overlap)})
    return results

# Temperature scaling: fit scalar T on held-out labeled validation logits when available.
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))
    def forward(self, logits):
        return logits / self.log_temperature.exp().clamp(0.05, 10.0)
    def fit(self, logits, labels, max_iter=100):
        logits = torch.tensor(logits, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.long)
        opt = torch.optim.LBFGS([self.log_temperature], lr=0.01, max_iter=max_iter)
        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(self(logits), labels)
            loss.backward()
            return loss
        opt.step(closure)
        return float(self.log_temperature.exp().detach().item())

temperature_scaler = TemperatureScaler()
CALIBRATION_TEMPERATURE = 1.0
# Optional calibration data CSV columns: logits (JSON list) and label.
CALIBRATION_CSV = ""
if CALIBRATION_CSV and os.path.exists(CALIBRATION_CSV):
    cal = pd.read_csv(CALIBRATION_CSV)
    if {"logits", "label"}.issubset(cal.columns):
        cal_logits = np.vstack(cal["logits"].apply(json.loads).values)
        CALIBRATION_TEMPERATURE = temperature_scaler.fit(cal_logits, cal["label"].astype(int).values)
print("Calibration temperature:", CALIBRATION_TEMPERATURE,
      "(1.0 means no validation calibration was fitted)")

# ------------------------- MAIN QUERY API -------------------------
def ask_scientific_question(query, save=True):
    if not chunks:
        raise RuntimeError(f"No indexed scientific documents. Add PDFs to {DATA_DIR} and rerun the cell.")
    text_ev, visual_ev = retrieve_evidence(query)
    G = build_evidence_graph(text_ev)
    graph_summary = graph_attention_summary(G)
    context = make_evidence_context(text_ev, visual_ev, graph_summary)
    answer = qwen_generate(query, context, visual_ev)
    claim_checks = verify_claims(answer, [x["text"] for x in text_ev])
    # Conservative evidence sufficiency indicator; it is not a clinical/probability guarantee.
    supported = [x for x in claim_checks if x["nli_label"] in ("entailment", "entails")]
    contradicted = [x for x in claim_checks if "contrad" in x["nli_label"]]
    mean_claim_conf = float(np.mean([x["nli_confidence"] for x in claim_checks])) if claim_checks else 0.0
    if not text_ev:
        status = "INSUFFICIENT_EVIDENCE"
    elif contradicted:
        status = "CONFLICTING_EVIDENCE"
    elif nli_model is not None and claim_checks and len(supported) < max(1, math.ceil(0.5*len(claim_checks))):
        status = "LOW_EVIDENCE_SUPPORT"
    else:
        status = "EVIDENCE_RETRIEVED_REQUIRES_SCIENTIFIC_REVIEW"
    result = {
        "query": query, "answer": answer, "evidence_status": status,
        "mean_claim_verification_score": mean_claim_conf,
        "claim_verification": claim_checks,
        "text_evidence": text_ev, "visual_evidence": visual_ev,
        "knowledge_graph": graph_summary,
        "calibration_temperature": CALIBRATION_TEMPERATURE
    }
    if save:
        safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", query[:45]).strip("_") or "query"
        with open(os.path.join(OUTPUT_DIR, f"answer_{safe}.json"), "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
    print("\n" + "="*90 + "\nQUESTION:", query)
    print("\nANSWER:\n", answer)
    print("\nEVIDENCE STATUS:", status)
    print("\nCLAIM VERIFICATION:")
    for c in claim_checks:
        print("-", c["nli_label"], f"({c['nli_confidence']:.3f})", c["claim"][:240])
    print("\nSOURCES:")
    for i, ev in enumerate(text_ev, 1):
        print(f"[E{i}] {ev['document']} | page {ev['page']} | {ev['section']}")
    for v in visual_ev:
        print("[FIGURE]", v["document"], "| page", v["page"], "|", v["image_path"])
    return result

# --------------------- STAGE 6: SPIQA EVALUATION ------------------
def load_spiqa_jsonl(path):
    rows = []
    if not path or not os.path.exists(path):
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                x = json.loads(line)
                q = x.get("question") or x.get("query")
                a = x.get("answer") or x.get("answers")
                if isinstance(a, list):
                    a = " ".join(map(str, a))
                if q and a:
                    rows.append({"question":str(q), "reference_answer":str(a)})
            except Exception:
                pass
    return rows

def token_f1(prediction, reference):
    p = re.findall(r"\w+", prediction.lower())
    r = re.findall(r"\w+", reference.lower())
    if not p or not r:
        return 0.0
    common = Counter(p) & Counter(r)
    overlap = sum(common.values())
    precision, recall = overlap/len(p), overlap/len(r)
    return 2*precision*recall/max(1e-12, precision+recall)

def evaluate_spiqa(path=SPIQA_JSONL, limit=100):
    qa = load_spiqa_jsonl(path)
    if not qa:
        print("SPIQA evaluation skipped: set SPIQA_JSONL to a QA JSONL file.")
        return None
    qa = qa[:limit]
    rows = []
    for i, item in enumerate(qa, 1):
        try:
            out = ask_scientific_question(item["question"], save=False)
            pred = out["answer"]
            rows.append({"question":item["question"], "prediction":pred,
                         "reference":item["reference_answer"],
                         "token_f1":token_f1(pred, item["reference_answer"]),
                         "retrieved_evidence_count":len(out["text_evidence"]),
                         "claim_check_count":len(out["claim_verification"])})
            print(f"SPIQA {i}/{len(qa)} | token-F1={rows[-1]['token_f1']:.3f}")
        except Exception as e:
            rows.append({"question":item["question"], "prediction":"ERROR: "+str(e),
                         "reference":item["reference_answer"], "token_f1":0.0,
                         "retrieved_evidence_count":0, "claim_check_count":0})
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUTPUT_DIR, "SPIQA_evaluation_predictions.csv"), index=False)
    metrics = {"samples":len(df), "mean_token_f1":float(df["token_f1"].mean()),
               "mean_retrieved_evidence":float(df["retrieved_evidence_count"].mean())}
    with open(os.path.join(OUTPUT_DIR, "SPIQA_evaluation_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    print("SPIQA metrics:", metrics)
    return df, metrics

# ------------------------- ABLATION STUDY -------------------------
def ablation_retrieval(query, reference_answer=None):
    """Compare sparse-only, dense-only and hybrid RRF retrieval rankings."""
    if not chunks:
        raise RuntimeError("No corpus indexed.")
    sparse_ids = []
    if bm25:
        scores = bm25.get_scores(re.findall(r"\w+", query.lower()))
        sparse_ids = np.argsort(scores)[::-1][:TOP_K_RERANK].tolist()
    qemb = text_encoder.encode([query], normalize_embeddings=True).astype("float32")
    dense_ids = []
    if text_index is not None:
        _, ids = text_index.search(qemb, min(TOP_K_RERANK, len(chunks)))
        dense_ids = [int(i) for i in ids[0] if i >= 0]
    hybrid_ids = reciprocal_rank_fusion([sparse_ids, dense_ids])[:TOP_K_RERANK]
    methods = {"BM25_sparse":sparse_ids, "BGE_M3_dense":dense_ids, "Hybrid_RRF":hybrid_ids}
    rows = []
    for method, ids in methods.items():
        retrieved = " ".join(chunks[i]["text"] for i in ids)
        rows.append({"method":method, "retrieved_count":len(ids),
                     "reference_token_overlap": token_f1(retrieved, reference_answer)
                         if reference_answer else np.nan,
                     "top_document":chunks[ids[0]]["document"] if ids else "",
                     "top_page":chunks[ids[0]]["page"] if ids else ""})
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUTPUT_DIR, "retrieval_ablation.csv"), index=False)
    return df

# ---------------------- SAVE GRAPH AND SUMMARY --------------------
def export_knowledge_graph(result, filename="knowledge_graph"):
    kg = result.get("knowledge_graph", {})
    pd.DataFrame(kg.get("nodes", [])).to_json(
        os.path.join(OUTPUT_DIR, filename+"_nodes.json"), orient="records", indent=2)
    pd.DataFrame(kg.get("edges", [])).to_json(
        os.path.join(OUTPUT_DIR, filename+"_edges.json"), orient="records", indent=2)

# ---------------------------- RUN --------------------------------
summary = {
    "pdf_count":len(pdf_files), "passage_count":len(chunks),
    "figure_count":len(valid_figures), "text_embedding_shape":list(text_embeddings.shape),
    "visual_embedding_shape":list(figure_embeddings.shape), "device":DEVICE,
    "output_directory":OUTPUT_DIR
}
with open(os.path.join(OUTPUT_DIR, "pipeline_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("\nPIPELINE INITIALIZATION COMPLETE")
print(json.dumps(summary, indent=2))
print("\nExample:")
print('result = ask_scientific_question("What methods and datasets are used in the paper?")')
print('ablation_df = ablation_retrieval("What is the proposed method?")')
print('evaluate_spiqa()  # if SPIQA_JSONL is configured')
print("Outputs are saved in:", OUTPUT_DIR)
